In [32]:
import glob, os, subprocess, zipfile, getpass
import numpy as np, pandas as pd

LIMIT = 600
if "KGAT_214173c4c6349afd02a57d764394c583" not in os.environ:
    os.environ["KGAT_214173c4c6349afd02a57d764394c583"] = getpass.getpass("KGAT_214173c4c6349afd02a57d764394c583: ")

sub = pd.read_csv("data/subset_ids.csv")
# take half pneumonia, half normal, so a small LIMIT stays balanced
sub = pd.concat([sub[sub.Target == 1].head(LIMIT // 2), sub[sub.Target == 0].head(LIMIT // 2)])
os.makedirs("data/images", exist_ok=True)

fails = 0
for i, pid in enumerate(sub.patientId):
    dcm = f"data/images/{pid}.dcm"
    if os.path.exists(dcm):
        continue
    r = subprocess.run(["kaggle", "competitions", "download", "-c", "rsna-pneumonia-detection-challenge",
                        "-f", f"stage_2_train_images/{pid}.dcm", "-p", "data/images"],
                       capture_output=True, text=True)
    z = dcm + ".zip"
    if os.path.exists(z):
        with zipfile.ZipFile(z) as zf: zf.extractall("data/images")
        os.remove(z)
    if not os.path.exists(dcm):
        fails += 1
        print("FAILED:", pid, (r.stdout + r.stderr)[-300:])
        if fails >= 3: print("Stopping. Send me this message."); break
    elif i % 20 == 0:
        print("downloaded", i + 1)
print("files on disk:", len(glob.glob("data/images/*.dcm")))

files on disk: 600


In [33]:
import pydicom
from PIL import Image
from sklearn.model_selection import train_test_split

os.makedirs("data/processed", exist_ok=True)
rows = []
for f in sorted(glob.glob("data/images/*.dcm")):
    pid = os.path.basename(f)[:-4]
    a = pydicom.dcmread(f).pixel_array.astype("float32")
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)
    Image.fromarray((a * 255).astype("uint8")).resize((224, 224)).save(f"data/processed/{pid}.png")
    rows.append(pid)

labels = pd.read_csv("data/subset_ids.csv").set_index("patientId").Target
df = pd.DataFrame({"patientId": rows})
df["label"] = df.patientId.map(labels).astype(int)
print(df.label.value_counts().to_dict())

tr, te = train_test_split(df, test_size=0.2, stratify=df.label, random_state=42)
tr, va = train_test_split(tr, test_size=0.125, stratify=tr.label, random_state=42)
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")]) \
  .to_csv("data/splits.csv", index=False)
print(len(tr), "train |", len(va), "val |", len(te), "test")

{1: 300, 0: 300}
420 train | 60 val | 120 test
